# Radio Survey Pipelines I
## From raw LoTSS FITS cutouts to a defensible data pipeline

**90 minutes · analysis, scaling, imbalance and augmentation · no model training**

The archive contains 8,805 real LoTSS DR2 cutouts from the Horton et al.
visual-classification catalogue. It keeps sources with one unambiguous broad
label: FRI, FRII, Hybrid, Spiral or Relaxed double.

The labels are curated; the pixel arrays are not. The supplied FITS values
have not been normalised, stretched, resized or converted to PNG.

### Guiding question

> What information should reach the model, and which transformations can
> change the image without changing the morphology answer?

This is an observational laboratory rather than a programming exercise.
The cells below do the routine loading and plotting. The useful work is to
change one control at a time, compare the resulting views, and connect each
numerical operation to a scientific consequence. At the end the selected
pipeline is exported for Session II.

**Working pattern.** A cell with a control near its top is designed to be
rerun. Start from the displayed default, then alter a single setting. This
gives a more informative comparison than changing several choices at once.

In [ ]:
#@title Set up the notebook
%pip -q install astropy

from pathlib import Path
import hashlib, json, random, tarfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
drive.mount("/content/drive")
course_dir = Path("/content/drive/MyDrive/Granada School")
course_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title Locate and verify the LoTSS classroom archive
# Before opening this notebook, drag the supplied archive into:
# My Drive/Granada School/Granada_School_LoTSS_raw.tar.gz
ARCHIVE_FILENAME = "Granada_School_LoTSS_raw.tar.gz" #@param {type:"string"}
EXPECTED_SHA256 = "970d2fdcad11fbbcffd8c74fca8e2dc491adb253bd92e2913e5511e3d3046211"

archive_path = course_dir / ARCHIVE_FILENAME
data_root = Path("/content/granada_lotss")

if len(EXPECTED_SHA256) != 64:
    raise RuntimeError("The published notebook must contain the archive checksum.")

if not archive_path.exists():
    raise FileNotFoundError(
        "The supplied archive was not found. Drag "
        f"{ARCHIVE_FILENAME} into My Drive/Granada School, then rerun this cell."
    )
print(f"Found classroom archive in Google Drive: {archive_path.name}")

hasher = hashlib.sha256()
with archive_path.open("rb") as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        hasher.update(chunk)
digest = hasher.hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(f"Archive checksum mismatch: {digest}")

if not (data_root / "manifest.csv").is_file():
    print("Extracting the verified archive...")
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall("/content", filter="data")

manifest = pd.read_csv(data_root / "manifest.csv")
expected_counts = {
    "FRI": 2349,
    "FRII": 5660,
    "Hybrid": 337,
    "Spiral": 182,
    "Relaxed double": 277,
}
assert len(manifest) == 8805
assert manifest.source_id.nunique() == 8805
assert manifest.label.value_counts().to_dict() == expected_counts
assert all((data_root / path).is_file() for path in manifest.fits_path)
print("Verified 8,805 raw FITS cutouts.")
display(manifest.head(3))

## Checkpoint 1 — What dataset do we actually have? (15 minutes)

Begin with the manifest rather than loading 8,805 arrays into memory. The
split is fixed by source before any augmentation: all views of a source must
remain in the same split. Otherwise, a rotated or reflected training image
could leak information about a validation or test image.

The bar chart is deliberately a count plot, not evidence of astrophysical
prevalence. It describes this curated labelled sample. Use the controls to
switch between linear and logarithmic vertical axes, then notice how the
rare classes become visible without becoming less rare.

**Investigate.** Run the cell first with `SHOW_SPLIT_TABLE=True` and
`LOG_COUNT_AXIS=False`. Check that every class occurs in every split, and
that the split ratios are similar rather than identical by accident. Then
set `LOG_COUNT_AXIS=True` and rerun. The scientific fact has not changed;
only the visual scale has. This is a useful reminder that an effective plot
makes minority classes inspectable without correcting the imbalance.

**What the code does.** `groupby` counts rows for every `(split, label)`
pair and `unstack` turns that long list into the displayed table. The next
lines calculate the total count, fraction and majority-to-class ratio. The
last block draws the bar chart, with the optional logarithmic axis applied
only after the data values have been counted.

In [ ]:
class_order = ["FRI", "FRII", "Hybrid", "Spiral", "Relaxed double"]
SHOW_SPLIT_TABLE = True #@param {type:"boolean"}
LOG_COUNT_AXIS = False #@param {type:"boolean"}

split_table = (
    manifest.groupby(["split", "label"]).size()
    .unstack(fill_value=0).reindex(columns=class_order)
)
class_counts = manifest.label.value_counts().reindex(class_order)
class_fractions = class_counts / class_counts.sum()
if SHOW_SPLIT_TABLE:
    display(split_table)
display(pd.DataFrame({
    "sources": class_counts,
    "fraction": class_fractions,
    "majority_to_class_ratio": class_counts.max() / class_counts,
}))

fig, ax = plt.subplots(figsize=(8, 4))
class_counts.plot.bar(ax=ax, color="#19A7AE")
if LOG_COUNT_AXIS:
    ax.set_yscale("log")
ax.set(title="Natural LoTSS class imbalance", ylabel="sources", xlabel="")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()

### Reading the imbalance

The following baseline is not a model to improve; it is a warning about an
easy-to-misread metric. It predicts the commonest class for every source.
Its accuracy is therefore the FRII fraction, despite having zero recall for
every other morphology. In this practical, **macro-F1** is the selection
metric because it gives every class one vote through its own precision and
recall. Balanced accuracy, the mean recall, will be reported alongside it.

When the printed accuracy looks respectable, ask which sources have actually
been recovered. That distinction will matter again after training.

**What the code does.** `idxmax` finds the commonest label and the division
by the total number of rows gives the accuracy of always choosing it. There
is no fitting or prediction here. It is an analytical baseline that shows
why later cells must report per-class quantities as well as accuracy.

In [ ]:
majority_class = class_counts.idxmax()
majority_accuracy = class_counts.max() / class_counts.sum()
print(f"Majority-only rule: always predict {majority_class}")
print(f"Accuracy: {majority_accuracy:.3f}")
print("Minority recalls: all zero")

### Optional catalogue audit

Class counts describe only one aspect of the sample. The manifest also
records cutout angular size, array dimensions and secondary visual flags
from the Horton catalogue. Choose one view. These fields can help diagnose
selection effects and ambiguous morphology, but the secondary visual flags
must not be handed to the image classifier as input features: they contain
information supplied by the same classification process that defined the
target.

**Investigate.** Change `AUDIT_VIEW`, rerun, and ask a different question
each time. `cutout_size` asks whether classes were framed comparably;
`array_dimensions` reveals the technical variety of the image arrays; and
`secondary_morphology` shows associations in the catalogue that must not be
treated as independent predictor variables. Do not try to infer causation
from these descriptive plots.

**What the code does.** The first branch uses a box plot to summarise the
distribution of angular cutout sizes. The second counts `(shape_y, shape_x)`
combinations. The final branch finds columns beginning `morph_`, calculates
their mean within each broad class, and visualises those fractions as a
heatmap. The `if`/`elif` structure ensures that only the chosen view runs.

In [ ]:
AUDIT_VIEW = "cutout_size" #@param ["cutout_size", "array_dimensions", "secondary_morphology"]

if AUDIT_VIEW == "cutout_size":
    plt.figure(figsize=(9, 4))
    sns.boxplot(
        data=manifest, x="label", y="cutout_arcmin",
        order=class_order, showfliers=False,
    )
    plt.xticks(rotation=25)
    plt.xlabel("")
    plt.ylabel("cutout size (arcmin)")
    plt.title("Cutout-size distributions by class")
    plt.tight_layout()
elif AUDIT_VIEW == "array_dimensions":
    shape_counts = (
        manifest.groupby(["shape_y", "shape_x"]).size()
        .sort_values(ascending=False).head(15)
    )
    display(shape_counts.rename("sources").to_frame())
else:
    morphology_columns = [
        name for name in manifest.columns if name.startswith("morph_")
    ]
    feature_fraction = (
        manifest.groupby("label")[morphology_columns].mean()
        .reindex(class_order)
    )
    plt.figure(figsize=(12, 4))
    sns.heatmap(
        feature_fraction, cmap="mako", vmin=0, vmax=1,
        cbar_kws={"label": "fraction of sources"},
    )
    plt.xlabel("secondary Horton flag")
    plt.ylabel("broad class")
    plt.title("Secondary visual features by broad class")
    plt.tight_layout()

catalogue_observation = ""
print("Observation to discuss:", catalogue_observation)

## Checkpoint 2 — Interrogate raw FITS evidence (15 minutes)

A FITS cutout is a measured array, not a finished picture. The first panel
intentionally uses a raw linear display, where a small number of bright
pixels can conceal diffuse emission. The histogram shows why: radio images
commonly have a concentrated background distribution with a long positive
tail from sources and artefacts.

Change `chosen_class` and `example_number`, rerun the cell, and compare the
metadata, histogram and image. The two display flags make it easy to focus
on one kind of evidence at a time. Cleaning non-finite values is performed
consistently afterwards; the finite fraction remains visible in the summary
so that the operation is never silent.

**Investigate.** Start with `Spiral`, then choose an FRII and a Hybrid; set
`example_number` to a different non-negative integer within a class and
rerun. Toggle one display flag at a time. Compare the apparent morphology
in the raw linear image with the histogram's dynamic range and with the
header values (`BUNIT`, `BMAJ`, `BMIN`). This establishes what the array is
before any display stretch makes its structure easier to see.

**What the code does.** The cell selects one training-manifest row, opens
only its FITS primary array, and copies its header. `np.isfinite` creates a
pixel-validity mask used both for the finite fraction and for the histogram.
The modulo operation on `example_number` keeps the requested index inside
the available rows of the chosen class. No rescaling occurs in this cell.

In [ ]:
chosen_class = "Spiral" #@param ["FRI", "FRII", "Hybrid", "Spiral", "Relaxed double"]
example_number = 0 #@param {type:"integer"}
SHOW_RAW_LINEAR_IMAGE = True #@param {type:"boolean"}
SHOW_PIXEL_HISTOGRAM = True #@param {type:"boolean"}
candidates_for_class = manifest[(manifest.split == "train") &
                                (manifest.label == chosen_class)]
chosen = candidates_for_class.iloc[example_number % len(candidates_for_class)]
chosen_path = data_root / chosen.fits_path

with fits.open(chosen_path, memmap=True) as hdul:
    raw = np.asarray(hdul[0].data, dtype=np.float32)
    header = hdul[0].header.copy()

finite = np.isfinite(raw)
raw_summary = pd.Series({
    "source": chosen.source_id,
    "class": chosen.label,
    "shape": str(raw.shape),
    "dtype": str(raw.dtype),
    "finite fraction": finite.mean(),
    "minimum": np.nanmin(raw),
    "median": np.nanmedian(raw),
    "maximum": np.nanmax(raw),
    "BUNIT": header.get("BUNIT", "not recorded"),
    "BMAJ": header.get("BMAJ", "not recorded"),
    "BMIN": header.get("BMIN", "not recorded"),
})
display(raw_summary)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
if SHOW_RAW_LINEAR_IMAGE:
    axes[0].imshow(raw, origin="lower", cmap="gray")
    axes[0].set_title("Raw linear display")
else:
    axes[0].axis("off")
if SHOW_PIXEL_HISTOGRAM:
    axes[1].hist(raw[finite], bins=100, log=True, color="#1B4965")
    axes[1].set_title("Finite pixel distribution")
    axes[1].set_xlabel(header.get("BUNIT", "pixel value"))
else:
    axes[1].axis("off")
for axis in axes:
    axis.grid(False)
plt.tight_layout()

### A fixed cleaning convention for this workshop

The pipeline replaces NaN and infinite values with zero before scaling. This
is a pragmatic convention, not a statement that a missing measurement is
physically zero. It is acceptable here only because the notebook also shows
the finite-pixel fraction and applies exactly the same rule to every split.
A survey analysis with substantial invalid regions would normally retain a
mask or model the missingness explicitly.

**What the code does.** `np.nan_to_num` constructs `clean`, a float32 copy
in which NaN, positive infinity and negative infinity are assigned zero.
Every later display and model-input function uses this same `clean` array.
The original `raw` array remains available above for inspection, so the
cleaning convention does not overwrite the observation.

In [ ]:
clean = np.nan_to_num(
    raw, nan=0.0, posinf=0.0, neginf=0.0
).astype(np.float32)

## Checkpoint 3 — Scaling is part of the experiment (20 minutes)

Compare raw linear values, min–max, percentile clipping, asinh and the
standard Astropy z-scale. All panels remain greyscale; a stretch is not a
colour map.

These methods answer different practical questions. Min–max retains the
ordering of every finite value but lets two extreme pixels determine the
contrast. Percentile clipping limits that influence by deliberately
saturating the tails. Asinh is useful when bright compact structure and
fainter extended emission must remain visible together. Z-scale estimates a
robust display interval from the image distribution and is widely used for
astronomical inspection. Applied independently to every source, however, it
removes a simple common intensity scale across the sample.
Use `SHOW_ALL_STRETCHES` to turn the full comparison panel on and off. Then
choose a candidate model-input scaling in the following cell and use the
gallery to compare that one choice across morphology classes.

**Investigate.** Leave `SHOW_ALL_STRETCHES=True` for the first run. Focus on
a diffuse feature and on the brightest compact feature: which methods reveal
each, and which saturate or suppress it? Set `SHOW_SCALING_SUMMARY=True` if
the numerical limits help the comparison. Then change only
`SCALING_CHOICE` in the next cell and rerun the gallery. A sensible choice
should work across several source types, not merely make one example look
attractive.

**What the code does.** The four small functions map the cleaned array into
a display range. Min-max and percentile scaling choose explicit lower and
upper limits; asinh applies a nonlinear compression after percentile scaling;
`ZScaleInterval` estimates limits from the image distribution. `views` holds
every result for the comparison grid. The summary reports quantities derived
from those transformed arrays, not new measurements of the sky.

In [ ]:
def minmax_scale(image):
    low, high = np.min(image), np.max(image)
    if high <= low:
        return np.zeros_like(image)
    return np.clip((image - low) / (high - low), 0, 1)

def percentile_scale(image, lower=0.5, upper=99.5):
    low, high = np.percentile(image, [lower, upper])
    if high <= low:
        return np.zeros_like(image)
    return np.clip((image - low) / (high - low), 0, 1)

def asinh_scale(image, softening=0.08):
    scaled = percentile_scale(image)
    return np.arcsinh(scaled / softening) / np.arcsinh(1 / softening)

def zscale_image(image):
    return np.asarray(ZScaleInterval()(image), dtype=np.float32)

def scale_image(image, method):
    routes = {
        "minmax": minmax_scale,
        "percentile": percentile_scale,
        "asinh": asinh_scale,
        "zscale": zscale_image,
    }
    return routes[method](image)

SHOW_ALL_STRETCHES = True #@param {type:"boolean"}
SHOW_SCALING_SUMMARY = False #@param {type:"boolean"}

views = {
    "raw linear": clean,
    "min–max": minmax_scale(clean),
    "percentile": percentile_scale(clean),
    "asinh": asinh_scale(clean),
    "z-scale": zscale_image(clean),
}

if SHOW_ALL_STRETCHES:
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    for axis, (name, image) in zip(axes, views.items()):
        axis.imshow(image, origin="lower", cmap="gray")
        axis.set_title(name)
        axis.axis("off")
    plt.tight_layout()

scaling_summary = pd.DataFrame({
    name: {
        "median": np.median(image),
        "p99": np.percentile(image, 99),
        "fraction at lower limit": np.mean(image <= np.min(image)),
        "fraction at upper limit": np.mean(image >= np.max(image)),
    }
    for name, image in views.items()
}).T
if SHOW_SCALING_SUMMARY:
    display(scaling_summary)

### How to interpret the comparison

A single extreme pixel controls min-max scaling. Percentile methods
deliberately saturate values outside their limits. Z-scale adapts contrast
to each source, which is useful when the target is morphology, but weakens
direct comparison of absolute brightness from one source to another. No
stretch is universally correct: the important point is that the selected
transformation is declared, applied consistently, and evaluated on held-out
sources processed in the same way.

### Choose a model-input scaling route

Z-scale is the recommended workshop default because the target is broad
morphology and the cutouts span a wide range of surface-brightness
distributions. The control remains open so that you can compare another
defensible route. The choice will be written into the Session I contract and
used unchanged in Session II.

**Investigate.** Keep `zscale` for a reference run, then select one
alternative and rerun the gallery with `SHOW_CLASS_GALLERY=True`. The two
examples per class are deliberately held fixed by the seed, so a changed
panel reflects the scaling choice rather than a different source. Return to
`zscale` unless the comparison gives a clear reason to use another policy.

**What the code does.** The Colab dropdown assigns one string to
`SCALING_CHOICE`. The gallery samples two training sources per label with a
fixed random seed, reads each FITS array, applies the selected function via
`scale_image`, and displays the result in greyscale. This is still a visual
audit; no images are resized or passed into a network in Session I.

In [ ]:
SCALING_CHOICE = "zscale" #@param ["zscale", "percentile", "asinh", "minmax"]
print("Selected model-input scaling:", SCALING_CHOICE)

In [ ]:
# Inspect two examples per class before generalising from one source.
SHOW_CLASS_GALLERY = True #@param {type:"boolean"}
examples = (
    manifest[manifest.split == "train"]
    .groupby("label", group_keys=False)
    .sample(n=2, random_state=SEED)
)
if SHOW_CLASS_GALLERY:
    fig, axes = plt.subplots(5, 2, figsize=(7, 14))
    for row_number, label in enumerate(class_order):
        for column, (_, item) in enumerate(
            examples[examples.label == label].iterrows()
        ):
            image = fits.getdata(data_root / item.fits_path).astype(np.float32)
            image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
            axes[row_number, column].imshow(
                scale_image(image, SCALING_CHOICE),
                origin="lower", cmap="gray", vmin=0, vmax=1
            )
            axes[row_number, column].set_title(label)
            axes[row_number, column].axis("off")
    plt.tight_layout()

## Checkpoint 4 — Augmentation is an invariance claim (20 minutes)

Inspect candidate changes on the currently selected real source. This is an
invariance test: a useful augmentation should change nuisance variation
while retaining the morphology label. Use the display controls to turn the
candidate grid and the absolute-difference view on and off.

The key distinction is between mathematical validity and scientific
validity. A library can rotate, crop or perturb an array without error. That
does not establish that the transformed observation retains both the source
and its label. Orientation is not part of the five broad Horton classes, so
right-angle rotations and reflections are a strong default. Translation,
cropping, noise injection and beam changes require limits tied to the
cutout, background and observing process.

**Investigate.** Begin with the full candidate grid. Turn on
`SHOW_TRANSLATION_DIFFERENCE` and increase `TRANSLATION_PIXELS` gradually.
The difference panel makes edge filling and displaced emission explicit.
Then change the raw-FITS source above and rerun this cell: a harmless-looking
transform for a compact source can be destructive for an extended double.
The only augmentation choices carried forward are `d4`, rotations only, or
none; translation, crop and fixed noise remain demonstrations rather than
hidden training operations.

**What the code does.** The cell first scales the selected cleaned array.
`np.rot90` and `np.fliplr` are exact array operations. `shift` interpolates
a translated image and fills newly exposed pixels with zero. The noise and
crop entries are intentionally simple visual examples. The optional magma
panel plots the absolute pixel-wise difference between the original and the
translated image.

In [ ]:
from scipy.ndimage import shift

SHOW_AUGMENTATION_GRID = True #@param {type:"boolean"}
SHOW_TRANSLATION_DIFFERENCE = False #@param {type:"boolean"}
TRANSLATION_PIXELS = 10 #@param {type:"integer"}

base = scale_image(clean, SCALING_CHOICE)
candidates = {
    "identity": base,
    "rotate 90 degrees": np.rot90(base),
    "horizontal reflection": np.fliplr(base),
    "bounded translation": shift(
        base, shift=(TRANSLATION_PIXELS, -TRANSLATION_PIXELS),
        order=1, mode="constant", cval=0
    ),
    "fixed display-space noise": np.clip(
        base + np.random.normal(0, 0.05, base.shape), 0, 1
    ),
    "central crop": base[
        base.shape[0]//4:-base.shape[0]//4,
        base.shape[1]//4:-base.shape[1]//4,
    ],
}
if SHOW_AUGMENTATION_GRID:
    fig, axes = plt.subplots(2, 3, figsize=(11, 7))
    for axis, (name, image) in zip(axes.flat, candidates.items()):
        axis.imshow(image, origin="lower", cmap="gray", vmin=0, vmax=1)
        axis.set_title(name)
        axis.axis("off")
    plt.tight_layout()
if SHOW_TRANSLATION_DIFFERENCE:
    difference = np.abs(base - candidates["bounded translation"])
    plt.figure(figsize=(5, 4))
    plt.imshow(difference, origin="lower", cmap="magma")
    plt.colorbar(label="absolute change in scaled intensity")
    plt.title("What the translation operation changed")
    plt.axis("off")
    plt.tight_layout()

**What the panels show.** Right-angle rotations and reflections are exact
pixel-grid operations. They are reasonable defaults because sky orientation
is not encoded in these broad morphology labels. A translation introduces
zero-filled edges and can move a lobe out of the cutout. A central crop can
remove the very extended structure needed to distinguish FR classes. The
fixed display-space noise shown here is a visual caution, not an observing
model: physical noise augmentation would be tied to a measured local RMS.

### Compose the training-view policy

A real augmentation pipeline is a sequence of independently sampled
operations, not a single named transform. Each enabled operation has a
probability: 1.0 means apply it to every training view; 0.25 means apply it
on roughly one in four requests. Multiple enabled operations may occur in
the same view, in the order shown. The default is conservative: D4 only.
Validation and test images remain unchanged under every policy.

**Investigate.** Begin with the default, then enable exactly one additional
perturbation at a modest probability. Rerun the preview and inspect several
draws, not just one attractive example. Translation and beam smoothing are
geometric/resolution claims; white and correlated noise are observational
claims. Flux scaling is an intensity-robustness test. With per-image z-scale
it is not a faithful simulation of an absolute flux-calibration error, so
keep its range mild. Compare a single addition against the D4 reference
before combining multiple operations.

**What the code does.** The policy dictionary is stored in the Session I
contract. A preview applies the same sequence that Session II will use:
D4, translation, white noise, beam-correlated noise, mild beam smoothing,
then a flux-scale perturbation. Each operation samples its own Bernoulli
probability. No
augmented files are written; a new view is generated only when a training
example is requested.

In [ ]:
from scipy.ndimage import gaussian_filter, shift

USE_D4 = True #@param {type:"boolean"}
P_D4 = 1.0 #@param {type:"number"}
USE_TRANSLATION = False #@param {type:"boolean"}
P_TRANSLATION = 0.20 #@param {type:"number"}
MAX_TRANSLATION_PIXELS = 6 #@param {type:"integer"}
USE_WHITE_NOISE = False #@param {type:"boolean"}
P_WHITE_NOISE = 0.20 #@param {type:"number"}
WHITE_NOISE_SIGMA = 0.02 #@param {type:"number"}
USE_CORRELATED_NOISE = False #@param {type:"boolean"}
P_CORRELATED_NOISE = 0.15 #@param {type:"number"}
CORRELATED_NOISE_SIGMA = 0.02 #@param {type:"number"}
CORRELATION_LENGTH_PIXELS = 1.0 #@param {type:"number"}
USE_BEAM_SMOOTHING = False #@param {type:"boolean"}
P_BEAM_SMOOTHING = 0.15 #@param {type:"number"}
BEAM_SIGMA_PIXELS = 0.75 #@param {type:"number"}
USE_FLUX_SCALE = False #@param {type:"boolean"}
P_FLUX_SCALE = 0.20 #@param {type:"number"}
MAX_FLUX_FRACTION = 0.10 #@param {type:"number"}
SHOW_POLICY_PREVIEW = True #@param {type:"boolean"}
PREVIEW_DRAWS = 6 #@param {type:"integer"}

augmentation_policy = {
    "d4": {"enabled": USE_D4, "probability": float(P_D4)},
    "translation": {"enabled": USE_TRANSLATION, "probability": float(P_TRANSLATION),
                    "max_pixels": int(MAX_TRANSLATION_PIXELS)},
    "white_noise": {"enabled": USE_WHITE_NOISE, "probability": float(P_WHITE_NOISE),
                    "sigma": float(WHITE_NOISE_SIGMA)},
    "correlated_noise": {"enabled": USE_CORRELATED_NOISE, "probability": float(P_CORRELATED_NOISE),
                         "sigma": float(CORRELATED_NOISE_SIGMA),
                         "correlation_length_pixels": float(CORRELATION_LENGTH_PIXELS)},
    "beam_smoothing": {"enabled": USE_BEAM_SMOOTHING, "probability": float(P_BEAM_SMOOTHING),
                       "sigma_pixels": float(BEAM_SIGMA_PIXELS)},
    "flux_scale": {"enabled": USE_FLUX_SCALE, "probability": float(P_FLUX_SCALE),
                   "max_fraction": float(MAX_FLUX_FRACTION)},
}

def probability(value):
    return float(np.clip(value, 0.0, 1.0))

def augmented_view(image, policy, rng):
    view = image.copy()
    if policy["d4"]["enabled"] and rng.random() < probability(policy["d4"]["probability"]):
        view = np.rot90(view, int(rng.integers(0, 4)))
        if rng.random() < 0.5:
            view = np.fliplr(view)
    if policy["translation"]["enabled"] and rng.random() < probability(policy["translation"]["probability"]):
        limit = max(0, int(policy["translation"]["max_pixels"]))
        view = shift(view, shift=tuple(rng.integers(-limit, limit + 1, size=2)),
                     order=1, mode="constant", cval=0.0)
    if policy["white_noise"]["enabled"] and rng.random() < probability(policy["white_noise"]["probability"]):
        view = view + rng.normal(0.0, max(0.0, policy["white_noise"]["sigma"]), view.shape)
    if policy["correlated_noise"]["enabled"] and rng.random() < probability(policy["correlated_noise"]["probability"]):
        noise = gaussian_filter(rng.normal(size=view.shape),
                                sigma=max(0.0, policy["correlated_noise"]["correlation_length_pixels"]))
        noise = noise / max(float(noise.std()), 1e-6)
        view = view + noise * max(0.0, policy["correlated_noise"]["sigma"])
    if policy["beam_smoothing"]["enabled"] and rng.random() < probability(policy["beam_smoothing"]["probability"]):
        view = gaussian_filter(view, sigma=max(0.0, policy["beam_smoothing"]["sigma_pixels"]))
    if policy["flux_scale"]["enabled"] and rng.random() < probability(policy["flux_scale"]["probability"]):
        fraction = max(0.0, policy["flux_scale"]["max_fraction"])
        view = view * rng.uniform(max(0.0, 1.0 - fraction), 1.0 + fraction)
    return np.clip(view, 0.0, 1.0).astype(np.float32)

display(augmentation_policy)
if SHOW_POLICY_PREVIEW:
    base = scale_image(clean, SCALING_CHOICE)
    rng = np.random.default_rng(SEED)
    draws = [base] + [augmented_view(base, augmentation_policy, rng)
                      for _ in range(max(1, int(PREVIEW_DRAWS)))]
    columns = 4
    rows = int(np.ceil(len(draws) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(12, 3.2 * rows))
    for number, (axis, image) in enumerate(zip(np.ravel(axes), draws)):
        axis.imshow(image, origin="lower", cmap="gray", vmin=0, vmax=1)
        axis.set_title("original" if number == 0 else f"policy draw {number}")
        axis.axis("off")
    for axis in np.ravel(axes)[len(draws):]:
        axis.axis("off")
    plt.tight_layout()

## Checkpoint 5 — Freeze and export the contract (20 minutes)

The recommended policy uses greyscale z-scale and random D4
rotations/reflections on training data only, but the controls above allow a
student to test a different declared combination. The single channel is
repeated three times only to meet the interface of ImageNet-pretrained
backbones; no colour information is created.

Run the final cell after choosing the scaling and augmentation controls. It
records the operational choices only; it is not a student worksheet. Session
II reads this small contract so that the training experiment uses the exact
display and augmentation policy explored here.

**What the code does.** The final dictionary is a provenance record: dataset
identity, class order, cleaning convention, scaling route, channel handling,
split rule and augmentation policy. It is written as JSON in Google Drive.
The next notebook validates key fields before it constructs a dataset, which
prevents an unnoticed mismatch between the visual and training pipelines.

In [ ]:
pipeline_contract = {
    "schema": "granada-lotss-pipeline-1.1",
    "dataset": "Horton et al. LoTSS DR2 visual classifications",
    "cohort_rows": 8805,
    "class_order": class_order,
    "raw_input": "unnormalised FITS primary array",
    "nan_policy": "replace non-finite values with zero; inspect finite fraction",
    "retained_nan_diagnostic": "finite fraction displayed for each inspected cutout",
    "scaling": SCALING_CHOICE,
    "source_channels": 1,
    "model_channels": 3,
    "channel_conversion": "repeat the scaled greyscale channel",
    "image_size": 224,
    "train_augmentation": augmentation_policy,
    "validation_augmentation": [],
    "test_augmentation": [],
    "split_rule": "fixed source-level stratified 70/15/15 split",
    "primary_metric": "macro_f1",
    "notes": "Controls selected during the visual data-pipeline laboratory.",
}
contract_path = course_dir / "granada_lotss_pipeline_contract.json"
contract_path.write_text(
    json.dumps(pipeline_contract, indent=2) + "\n"
)
display(pipeline_contract)
print("Saved for Session II:", contract_path)

### Before Session II

Keep the contract in Google Drive. In the next session, the selected scaling
is applied independently to every FITS array; augmentation is available only
to the training split. The important ideas to carry forward are: a stretch
changes what a network can easily see; augmentation asserts an invariance;
and class imbalance changes the meaning of headline accuracy.